# Random Forest and time-aware tuning

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "src").is_dir() and (folder / "data").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook inside the project folder.")
sys.path.insert(0, str(PROJECT_ROOT))

## Import model and evaluation helpers

In [4]:
import pandas as pd
from src.data_loader import load_model_dataset
from src.evaluation import regression_metrics
from src.features import FEATURE_COLUMNS, build_supervised_features
from src.models import make_baseline_models, tune_random_forest
from src.preprocessing import chronological_split

## Prepare the same chronological split used by the baseline

In [5]:
monthly = load_model_dataset()
supervised = build_supervised_features(monthly)
train, test = chronological_split(supervised, test_months=18)
print(f"Train: {len(train)} rows; holdout: {len(test)} rows")

Train: 110 rows; holdout: 18 rows


## Fit the fixed-parameter Random Forest baseline

In [6]:
models = make_baseline_models()
forest_model = models["Random Forest"]
forest_model.fit(train[FEATURE_COLUMNS], train["Target_Arrivals"])
forest_predictions = forest_model.predict(test[FEATURE_COLUMNS])
forest_metrics = regression_metrics(test["Target_Arrivals"], forest_predictions)
forest_metrics

{'MAE': 18873.361291891953,
 'RMSE': 23196.20083353999,
 'R2': 0.7555513686491601,
 'MAPE': 10.243544132895071}

## Tune a second forest using training data only

In [7]:
tuned_search = tune_random_forest(train)
print("Selected parameters:", tuned_search.best_params_)
print(f"Best cross-validation MAE: {-tuned_search.best_score_:,.2f} arrivals")

Selected parameters: {'model__max_depth': 12, 'model__min_samples_leaf': 1, 'model__n_estimators': 500}
Best cross-validation MAE: 40,210.95 arrivals


## Evaluate the tuned model on the untouched holdout

In [8]:
tuned_predictions = tuned_search.predict(test[FEATURE_COLUMNS])
tuned_metrics = regression_metrics(test["Target_Arrivals"], tuned_predictions)
tuned_metrics

{'MAE': 22022.43716034151,
 'RMSE': 26557.233389872625,
 'R2': 0.6795800486997392,
 'MAPE': 12.083608221511028}

## Compare both Random Forest results

In [9]:
pd.DataFrame(
    [
        {"Model": "Random Forest", **forest_metrics},
        {"Model": "Tuned Random Forest", **tuned_metrics},
    ]
).set_index("Model")

,MAE,RMSE,R2,MAPE
Model,,,,
Random Forest,18873.361292,23196.200834,0.755551,10.243544
Tuned Random Forest,22022.437160,26557.233390,0.679580,12.083608
